# Imports

In [1]:
import pickle

import numpy as np
import pandas as pd

from tqdm import tqdm

from sklearn.model_selection import StratifiedKFold, cross_val_predict

## Utils

In [2]:
def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        return pickle.load(file)

# Loading Datasets

In [3]:
X_train = pd.read_parquet('../data/X_train_raw.parquet')
y_train = pd.read_parquet('../data/y_train.parquet')

X_test = pd.read_parquet('../data/X_test_raw.parquet')

In [4]:
X_train.head()

,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population
id,,,,,,,,,,
0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,M,Red_Sequence
1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,M,Red_Sequence
2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,O/B,Blue_Cloud
3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,M,Red_Sequence
4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,M,Red_Sequence


In [5]:
X_test.head()

,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population
id,,,,,,,,,,
577347,120.719779,23.924249,23.668066,21.951680,21.086183,20.180032,19.202124,0.429042,G/K,Red_Sequence
577348,219.414419,42.171651,24.902933,22.338822,20.732163,19.860330,19.687691,0.867305,M,Red_Sequence
577349,173.568731,-1.756400,19.427591,18.474633,17.551314,16.570674,16.176765,0.224234,G/K,Blue_Cloud
577350,184.903993,-1.411074,23.121029,21.526855,20.670159,20.417633,20.699095,0.066507,G/K,Red_Sequence
577351,222.487816,15.381403,25.094282,22.643981,21.123173,19.439500,19.094158,0.977218,M,Red_Sequence


# Machine Learning

In [6]:
models = dict(
    lgbm=load_pickle('../models/layer_1/model_lightgbm.pkl'),
    cat=load_pickle('../models/layer_1/model_catboost.pkl'),
    xgb=load_pickle('../models/layer_1/model_xgboost.pkl'),
    hist=load_pickle('../models/layer_1/model_hist_gradient_boosting.pkl'),
    extra=load_pickle('../models/layer_1/model_extra_tree.pkl'),
    rf=load_pickle('../models/layer_1/model_random_forest.pkl'),
    # lda=load_pickle('../models/layer_1/model_lda.pkl'),
    # linear_svc=load_pickle('../models/layer_1/model_linear_svc.pkl'),
    lg=load_pickle('../models/layer_1/model_logistic_regression.pkl'),
    # mlp=load_pickle('../models/layer_1/model_mlp.pkl'),
    # qda=load_pickle('../models/layer_1/model_qda.pkl'),
    # ridge=load_pickle('../models/layer_1/model_ridge.pkl'),
    # sgd=load_pickle('../models/layer_1/model_sgdclassifier.pkl'),
    # trunsvd_knn=load_pickle('../models/layer_1/model_trunsvd_knn.pkl'),
)

## Train Dataset

In [7]:
cv = StratifiedKFold(shuffle=True, random_state=42, n_splits=5)

In [8]:
X_train_stacking = pd.DataFrame({})

In [9]:
for model_name, model in tqdm(models.items()):
    
    print(f"Predicting Train Dataset {model_name}")

    predictions = cross_val_predict(model, X_train, y_train.class_encoded, cv=cv, n_jobs=-1, method='predict_proba')
    X_train_stacking[[f'{model_name}_0', f'{model_name}_1', f'{model_name}_2']] = predictions

  0%|                                                                                                                                                                                         | 0/7 [00:00<?, ?it/s]

Predicting Train Dataset lgbm


 14%|████████████████████████▊                                                                                                                                                     | 1/7 [10:22<1:02:13, 622.23s/it]

Predicting Train Dataset cat


 29%|██████████████████████████████████████████████████▎                                                                                                                             | 2/7 [19:49<49:10, 590.08s/it]

...

 71%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▋                                                  | 5/7 [23:49<05:57, 178.53s/it]

Predicting Train Dataset rf


 86%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▊                         | 6/7 [51:50<11:29, 689.30s/it]

Predicting Train Dataset lg


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [54:30<00:00, 467.16s/it]


## Test Dataset

In [10]:
X_test_stacking = pd.DataFrame({})

In [11]:
for model_name, model in models.items():
    
    print(f"Predicting Test Dataset {model_name}")
    
    X_test_stacking[[f'{model_name}_0', f'{model_name}_1', f'{model_name}_2']] = model.predict_proba(X_test)

Predicting Test Dataset lgbm
Predicting Test Dataset cat
Predicting Test Dataset xgb
Predicting Test Dataset hist
Predicting Test Dataset extra
Predicting Test Dataset rf
Predicting Test Dataset lg


# Saving

In [12]:
X_train_stacking.to_parquet('../data/X_train_stacking_layer_one.parquet')
X_test_stacking.to_parquet('../data/X_test_stacking_layer_one.parquet')

In [13]:
X_train_stacking.head()

,lgbm_0,lgbm_1,lgbm_2,cat_0,cat_1,cat_2,xgb_0,xgb_1,xgb_2,hist_0,...,hist_2,extra_0,extra_1,extra_2,rf_0,rf_1,rf_2,lg_0,lg_1,lg_2
0,0.999946,0.000047,6.235225e-06,0.999971,0.000025,3.499764e-06,0.999863,0.000131,0.000006,9.999882e-01,...,2.170196e-06,0.939268,0.005067,0.055665,0.999938,0.000039,0.000023,0.986388,0.000051,1.356105e-02
1,0.988228,0.000484,1.128788e-02,0.986465,0.000105,1.343005e-02,0.988845,0.000337,0.010818,9.752932e-01,...,2.435544e-02,0.838575,0.003327,0.158099,0.970744,0.000420,0.028836,0.808375,0.003757,1.878684e-01
2,0.000006,0.999994,4.529885e-07,0.000006,0.999994,3.879918e-10,0.000007,0.999992,0.000001,8.011669e-07,...,6.045421e-08,0.005466,0.974345,0.020190,0.000004,0.999996,0.000000,0.000339,0.999661,9.150223e-19
3,0.999831,0.000159,9.526890e-06,0.999900,0.000099,3.615600e-07,0.999711,0.000280,0.000009,9.998010e-01,...,2.440324e-06,0.948374,0.004303,0.047323,0.999501,0.000452,0.000047,0.997360,0.001557,1.082721e-03
4,0.998633,0.001335,3.222033e-05,0.999030,0.000962,8.227142e-06,0.999295,0.000690,0.000015,9.977268e-01,...,1.867383e-05,0.924001,0.005180,0.070819,0.997856,0.001398,0.000746,0.986028,0.011410,2.561574e-03


In [14]:
X_test_stacking.head()

,lgbm_0,lgbm_1,lgbm_2,cat_0,cat_1,cat_2,xgb_0,xgb_1,xgb_2,hist_0,...,hist_2,extra_0,extra_1,extra_2,rf_0,rf_1,rf_2,lg_0,lg_1,lg_2
0,0.998938,0.000945,0.000117,0.997728,1.893188e-03,3.784761e-04,0.998359,0.001546,0.000096,0.998748,...,0.000071,0.682708,0.074696,0.242596,0.985949,0.006535,0.007515,0.976752,0.008796,0.014452
1,0.997533,0.002457,0.000010,0.997947,2.053297e-03,1.286922e-07,0.998815,0.001179,0.000007,0.992461,...,0.000008,0.931738,0.020059,0.048202,0.997601,0.002346,0.000053,0.993734,0.006230,0.000036
2,0.996906,0.000477,0.002618,0.999850,1.264084e-07,1.494327e-04,0.997993,0.000357,0.001650,0.994105,...,0.004532,0.589324,0.012476,0.398200,0.980884,0.002929,0.016187,0.806512,0.000821,0.192667
3,0.001207,0.000409,0.998383,0.000800,1.348342e-04,9.990656e-01,0.001168,0.000338,0.998494,0.000530,...,0.999053,0.070269,0.070010,0.859721,0.006972,0.004733,0.988295,0.073795,0.013885,0.912319
4,0.999827,0.000158,0.000015,0.999818,1.822315e-04,1.314618e-07,0.999859,0.000132,0.000009,0.999686,...,0.000026,0.942776,0.013396,0.043828,0.999693,0.000075,0.000232,0.999710,0.000287,0.000003


In [15]:
X_train.shape

(577347, 10)

In [16]:
X_test.shape

(247435, 10)

In [17]:
y_train.head()

,class,class_encoded
id,,
0,GALAXY,0
1,GALAXY,0
2,QSO,1
3,GALAXY,0
4,GALAXY,0
